# Recommendation & Inventory Intelligence Engine

This notebook demonstrates the end-to-end workflow across the 4 core pipelines:
1. **Data Ingestion & Feature Engineering**: Transaction baskets, inventory velocity, and cyclical temporal transforms (\$\sin/\cos\$).
2. **Spatial Representation (Hybrid Items & Customer Embeddings)**: Multi-modal item embeddings + Customer profile vectors mapped to a shared 64-d latent space.
3. **Recommendation Engines**:
   - **Item-to-Item Similarity**: Cosine distance retrieval on catalog items.
   - **Customer-to-Customer (Lookalike Collaborative Filtering)**: Recommending unpurchased items bought by similar customer peers.
   - **Seasonal & Trend Boosting**: Quarterly/weekly lift re-ranking.
4. **Inventory & Pricing Intelligence**: Statistical Safety Stock, Dynamic Reorder Points (ROP), and Price Elasticity of Demand (\$\text{PED}\$) per cluster.

## 1. Imports & Setup

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from src.ingestion.mssql_client import MSSQLIngestionClient
from src.features.temporal import add_temporal_cyclical_features, compute_seasonal_velocity_indices
from src.features.baskets import extract_transaction_baskets
from src.features.inventory_features import compute_daily_demand_metrics, compute_turnover_velocity
from src.embeddings.item2vec import Item2VecModel
from src.embeddings.hybrid import HybridEmbeddingBuilder
from src.embeddings.customer_embeddings import CustomerEmbeddingBuilder
from src.recommenders.item_to_item import ItemToItemRecommender
from src.recommenders.customer_lookalike import CustomerLookalikeRecommender
from src.recommenders.user_personalized import UserPersonalizedRecommender
from src.recommenders.seasonal_boosting import SeasonalBoostingEngine
from src.inventory_pricing.safety_stock import SafetyStockCalculator
from src.inventory_pricing.elasticity import PriceElasticityEngine

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("All modules loaded successfully!")

## 2. Ingestion & Feature Engineering

In [ ]:
client = MSSQLIngestionClient()
catalog_df = client.fetch_catalog()
tx_df = client.fetch_transactions(days_back=180)

tx_df_temporal = add_temporal_cyclical_features(tx_df, timestamp_col='timestamp')
demand_metrics_df = compute_daily_demand_metrics(tx_df_temporal, date_col='date')
inventory_state_df = compute_turnover_velocity(catalog_df, demand_metrics_df)
seasonal_indices_df = compute_seasonal_velocity_indices(tx_df_temporal)

print(f"Catalog items: {catalog_df.shape[0]} | Transactions: {tx_df.shape[0]}")

## 3. Spatial Representation: Item & Customer Latent Embeddings

In [ ]:
# 1. Train Item2Vec Skip-Gram
baskets = extract_transaction_baskets(tx_df_temporal)
item2vec = Item2VecModel(vector_size=32, epochs=30)
item2vec.fit(baskets)
i2v_embeddings = item2vec.get_all_embeddings()

# 2. Build Hybrid Item Embeddings (64-d)
hybrid_builder = HybridEmbeddingBuilder(target_dim=64)
item_embeddings = hybrid_builder.build_hybrid_vectors(items_df=catalog_df, item2vec_dict=i2v_embeddings)

# 3. Build Customer Embeddings in same latent space
cust_builder = CustomerEmbeddingBuilder(half_life_days=30.0)
customer_embeddings = cust_builder.build_all_customer_embeddings(tx_df_temporal, item_embeddings)

print(f"Item embeddings: {len(item_embeddings)} (64-d)")
print(f"Customer embeddings: {len(customer_embeddings)} (64-d)")

## 4. Customer-to-Customer Collaborative Recommendations (Lookalikes)

When a similar customer purchases an item, it is surfaced to peer customers who have not yet bought it.

In [ ]:
lookalike_engine = CustomerLookalikeRecommender(customer_embeddings)
sample_customer = list(customer_embeddings.keys())[0]

# 1. Find Lookalike Customer Peers
peers = lookalike_engine.find_lookalike_customers(sample_customer, top_k=5)
print(f"Top Lookalike Peers for {sample_customer}:")
for p in peers:
    print(f"  -> Peer: {p['lookalike_customer_id']} | Cosine Similarity: {p['similarity_score']:.4f}")

# 2. Recommend Items Purchased by Similar Peers
stock_map = {row['item_id']: row['current_stock'] for row in catalog_df.iter_rows(named=True)}
peer_recs = lookalike_engine.recommend_from_similar_customers(
    sample_customer,
    transactions_df=tx_df_temporal,
    top_k_items=5,
    n_lookalikes=5,
    stock_map=stock_map,
)

print(f"\nItems Recommended to {sample_customer} from Lookalike Purchases:")
for r in peer_recs:
    print(f"  Rank {r['rank']}: {r['item_id']} | Collaborative Score: {r['collaborative_score']:.4f} | Bought by: {r['recommended_via_lookalike_ids']}")

## 5. Inventory & Pricing Decision Layer

In [ ]:
stock_calc = SafetyStockCalculator(z_score=1.65)
inventory_results = stock_calc.batch_calculate(inventory_state_df)

elasticity_engine = PriceElasticityEngine(n_clusters=5)
cluster_map = elasticity_engine.cluster_items(item_embeddings)
cluster_ped = elasticity_engine.estimate_cluster_elasticity(tx_df_temporal, cluster_map=cluster_map)

print("Inventory Health Status:")
print(inventory_results['stock_status'].value_counts())

print("\nCluster Price Elasticity (PED):")
for cid, res in cluster_ped.items():
    print(f"  Cluster {cid}: PED = {res['ped']} | Action = {res['recommendation']}")

## 6. Summary & Key Findings

### Data Analysis Key Findings
- **Shared Metric Space**: Mapping both customers and items to 64-dimensional normalized vectors allows seamless dual retrieval: Customer $\leftrightarrow$ Item direct scoring and Customer $\leftrightarrow$ Customer lookalike search.
- **Peer Collaborative Filtering**: Customers with high embedding similarity share implicit taste clusters. Surfaces serendipitous recommendations from peer purchases that pure item-similarity would miss.
- **Inventory & Pricing Protection**: Safety Stock buffers against demand spikes caused by recommendation boosts, while Price Elasticity informs promotional discount thresholds.

### Next Steps
1. Run `src/database/schema.sql` in Supabase to create the `customer_embeddings` and `customer_lookalikes` tables.
2. Schedule `run_pipeline.py` to refresh both item and customer latent representations nightly.